# 06 — Full Label Audit (Step 1: Reproduce Training Dataset)

**Purpose:** independently reproduce the exact merged training dataset built in
`04_train_classifier.ipynb`, as a precondition for auditing all 153 ClinVar-derived
severity labels for the research paper.

**Scope of this notebook (step 1 only):**
- Load `data/training_variants.csv`, `data/sequence_features.csv`, `data/structural_features.csv`.
- Merge with the same inner joins on `variant_id` that notebook 04 uses.
- Apply the same `dropna(subset=['heme_distance_angstrom'])`.
- Build the same 9-column feature matrix.
- Load (not refit) `models/label_encoder.pkl` and `models/imputer.pkl` and apply them.
- Verify the merged row count and class distribution against `models/cv_scores.json`.

No predictions, no cross-validation, and no edits to existing notebooks/backend files happen here.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')

# Identical to notebook 04's FEATURE_COLS
FEATURE_COLS = [
    'blosum62_score',
    'conservation_score',
    'pssm_score',
    'position_entropy',
    'heme_distance_angstrom',
    'burial_binary',
    'is_interface_residue',
    'af2_confidence',
    'contact_count_wt',
]
print(f"{len(FEATURE_COLS)} training features: {FEATURE_COLS}")

9 training features: ['blosum62_score', 'conservation_score', 'pssm_score', 'position_entropy', 'heme_distance_angstrom', 'burial_binary', 'is_interface_residue', 'af2_confidence', 'contact_count_wt']


## Reproduce the merge exactly as in notebook 04

In [2]:
labels_df = pd.read_csv(DATA_DIR / 'training_variants.csv')
seq_df    = pd.read_csv(DATA_DIR / 'sequence_features.csv')
struct_df = pd.read_csv(DATA_DIR / 'structural_features.csv')

df = (
    labels_df[['variant_id', 'gene', 'position', 'severity_label']]
    .merge(seq_df,    on='variant_id', how='inner')
    .merge(struct_df, on='variant_id', how='inner')
)

print(f"Merged dataset: {len(df)} variants")
print("\nClass distribution:")
print(df['severity_label'].value_counts())
print("\nMissing values per feature:")
print(df[FEATURE_COLS].isnull().sum())

Merged dataset: 153 variants

Class distribution:
severity_label
mild      106
severe     27
benign     20
Name: count, dtype: int64

Missing values per feature:
blosum62_score            0
conservation_score        0
pssm_score                0
position_entropy          0
heme_distance_angstrom    0
burial_binary             0
is_interface_residue      0
af2_confidence            0
contact_count_wt          0
dtype: int64


In [3]:
# Drop rows where heme_distance is missing (position not in structure) — same as notebook 04
df = df.dropna(subset=['heme_distance_angstrom']).reset_index(drop=True)
print(f"After dropping rows with missing heme_distance: {len(df)} variants")

X = df[FEATURE_COLS].astype(float).values
y_raw = df['severity_label'].values
print(f"\nFeature matrix shape: {X.shape}")

After dropping rows with missing heme_distance: 153 variants

Feature matrix shape: (153, 9)


## Load (not refit) the saved label encoder and imputer

In [4]:
le      = joblib.load(MODELS_DIR / 'label_encoder.pkl')
imputer = joblib.load(MODELS_DIR / 'imputer.pkl')

print(f"Loaded label_encoder classes: {list(le.classes_)}")
print(f"Loaded imputer strategy: {imputer.strategy}")

# .transform (never .fit / .fit_transform) — these artifacts are frozen from notebook 04
y         = le.transform(y_raw)
X_imputed = imputer.transform(X)

print(f"\nLabel encoding observed in this audit: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"X_imputed shape: {X_imputed.shape}")
print(f"NaNs remaining after imputer.transform: {np.isnan(X_imputed).sum()}")

Loaded label_encoder classes: ['benign', 'mild', 'severe']
Loaded imputer strategy: mean

Label encoding observed in this audit: {'benign': np.int64(0), 'mild': np.int64(1), 'severe': np.int64(2)}
X_imputed shape: (153, 9)
NaNs remaining after imputer.transform: 0


## Verify against the recorded cv_scores.json

In [5]:
with open(MODELS_DIR / 'cv_scores.json') as f:
    recorded = json.load(f)

recorded_n           = recorded['n_training_variants']
recorded_class_counts = recorded['class_counts']

observed_n            = len(df)
observed_class_counts = df['severity_label'].value_counts().to_dict()

print("Recorded (cv_scores.json):")
print(f"  n_training_variants = {recorded_n}")
print(f"  class_counts         = {recorded_class_counts}")
print("\nObserved (this audit):")
print(f"  n_training_variants = {observed_n}")
print(f"  class_counts         = {observed_class_counts}")

n_match     = observed_n == recorded_n
class_match = observed_class_counts == recorded_class_counts

print("\n" + "=" * 60)
if n_match and class_match:
    print("MATCH CONFIRMED: audit dataset reproduces notebook 04 exactly.")
    print(f"  {observed_n} variants — "
          f"mild={observed_class_counts.get('mild', 0)}, "
          f"severe={observed_class_counts.get('severe', 0)}, "
          f"benign={observed_class_counts.get('benign', 0)}")
else:
    print("DISCREPANCY DETECTED — do not proceed without resolving this:")
    if not n_match:
        print(f"  Row count differs: observed={observed_n} vs recorded={recorded_n} "
              f"(delta={observed_n - recorded_n})")
    if not class_match:
        all_labels = set(observed_class_counts) | set(recorded_class_counts)
        for label in sorted(all_labels):
            o = observed_class_counts.get(label, 0)
            r = recorded_class_counts.get(label, 0)
            flag = "  <-- MISMATCH" if o != r else ""
            print(f"  {label}: observed={o} vs recorded={r}{flag}")
print("=" * 60)

Recorded (cv_scores.json):
  n_training_variants = 153
  class_counts         = {'mild': 106, 'severe': 27, 'benign': 20}

Observed (this audit):
  n_training_variants = 153
  class_counts         = {'mild': 106, 'severe': 27, 'benign': 20}

MATCH CONFIRMED: audit dataset reproduces notebook 04 exactly.
  153 variants — mild=106, severe=27, benign=20


# Step 2 — Out-of-fold cross-validated predictions

Using the 153-row `X_imputed` / `y` already built above, run the same
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` +
`Pipeline([('impute', SimpleImputer(strategy='mean')), ('clf', rf)])` structure
notebook 04 uses for CV, but via `cross_val_predict` so we keep the actual
out-of-fold predicted labels and probabilities per variant (not just aggregate
scores) — needed for the per-variant audit table in the next step.

Note: we pass `X` (not `X_imputed`) into the pipeline, mirroring notebook 04's CV
cell exactly — the pipeline's own `impute` step handles imputation per-fold. This
is intentional: notebook 04's saved `imputer.pkl` (used for `X_imputed` above) was
fit on the *full* dataset *after* CV, so feeding `X_imputed` into CV here would not
match notebook 04's actual CV methodology.

No new model artifacts are saved. No edits to existing cells.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_pipe = Pipeline([('impute', SimpleImputer(strategy='mean')), ('clf', rf)])

print("Running cross_val_predict (method='predict')...")
y_pred_oof = cross_val_predict(rf_pipe, X, y, cv=cv, method='predict')

print("Running cross_val_predict (method='predict_proba')...")
y_proba_oof = cross_val_predict(rf_pipe, X, y, cv=cv, method='predict_proba')

print(f"\ny_pred_oof shape:  {y_pred_oof.shape}")
print(f"y_proba_oof shape: {y_proba_oof.shape}")
print(f"predict_proba columns correspond to le.classes_: {list(le.classes_)}")

Running cross_val_predict (method='predict')...


Running cross_val_predict (method='predict_proba')...



y_pred_oof shape:  (153,)
y_proba_oof shape: (153, 3)
predict_proba columns correspond to le.classes_: ['benign', 'mild', 'severe']


In [7]:
# Pooled out-of-fold scores (one score across all 153 OOF predictions),
# vs. cv_scores.json's mean-of-5-fold-scores — different aggregation, same ballpark expected.
oof_macro_f1 = f1_score(y, y_pred_oof, average='macro')
oof_accuracy = accuracy_score(y, y_pred_oof)

with open(MODELS_DIR / 'cv_scores.json') as f:
    recorded = json.load(f)

print("Recorded in cv_scores.json (mean across 5 fold-level scores):")
print(f"  rf_macro_f1_mean = {recorded['rf_macro_f1_mean']:.4f}  (± {recorded['rf_macro_f1_std']:.4f})")
print(f"  rf_accuracy_mean = {recorded['rf_accuracy_mean']:.4f}")

print("\nThis audit (pooled score across all out-of-fold predictions):")
print(f"  oof_macro_f1     = {oof_macro_f1:.4f}")
print(f"  oof_accuracy     = {oof_accuracy:.4f}")

f1_diff  = oof_macro_f1 - recorded['rf_macro_f1_mean']
acc_diff = oof_accuracy - recorded['rf_accuracy_mean']
print(f"\nDelta (this audit − recorded): macro-F1 = {f1_diff:+.4f}, accuracy = {acc_diff:+.4f}")

within_1std = abs(f1_diff) <= recorded['rf_macro_f1_std']
print(f"Pooled macro-F1 within ±1 recorded std of fold-mean macro-F1: {within_1std}")

Recorded in cv_scores.json (mean across 5 fold-level scores):
  rf_macro_f1_mean = 0.4995  (± 0.1624)
  rf_accuracy_mean = 0.6533

This audit (pooled score across all out-of-fold predictions):
  oof_macro_f1     = 0.5020
  oof_accuracy     = 0.6536

Delta (this audit − recorded): macro-F1 = +0.0025, accuracy = +0.0003
Pooled macro-F1 within ±1 recorded std of fold-mean macro-F1: True


# Step 3 — Per-variant audit table

**Caveat on `wt_aa`/`mut_aa`:** the merged `df` built in Step 1 only carries
`['variant_id', 'gene', 'position', 'severity_label']` forward from `labels_df`
— that's exactly what notebook 04 keeps, since it only needs `severity_label`
for training. `wt_aa`/`mut_aa` are therefore *not* columns on `df`. They do
exist on `labels_df` (already loaded in Step 1), so this step brings them in
via an additional left-merge on `variant_id`, scoped only to this new
`audit_df` — `df`, `X`, `y`, `X_imputed`, `y_pred_oof`, `y_proba_oof` from
Steps 1–2 are not touched.

`severity_flip` is computed as `{true_label, predicted_label} == {'benign', 'severe'}`
— true only when the model predicted the opposite extreme of the true label.

In [8]:
print("Columns already in df:", list(df.columns))
print("wt_aa/mut_aa present in labels_df:", {'wt_aa', 'mut_aa'}.issubset(labels_df.columns))

audit_df = df.merge(labels_df[['variant_id', 'wt_aa', 'mut_aa']], on='variant_id', how='left')
assert len(audit_df) == len(df), "wt_aa/mut_aa merge changed row count — investigate"
assert audit_df[['wt_aa', 'mut_aa']].isnull().sum().sum() == 0, "Unexpected missing wt_aa/mut_aa after merge"
print(f"audit_df row count after wt_aa/mut_aa merge: {len(audit_df)}")

Columns already in df: ['variant_id', 'gene', 'position', 'severity_label', 'blosum62_score', 'conservation_score', 'position_entropy', 'pssm_score', 'af2_confidence', 'heme_distance_angstrom', 'burial_binary', 'residue_exposure', 'contact_count_wt', 'is_interface_residue', 'contact_map_delta', 'rmsd_angstrom']
wt_aa/mut_aa present in labels_df: True
audit_df row count after wt_aa/mut_aa merge: 153


In [9]:
# Decode integer labels back to strings via the loaded label_encoder (never refit)
true_label      = le.inverse_transform(y)
predicted_label = le.inverse_transform(y_pred_oof)

# y_proba_oof columns follow le.classes_ order — verified in Step 2 as ['benign', 'mild', 'severe']
classes = list(le.classes_)
proba_benign = y_proba_oof[:, classes.index('benign')]
proba_mild   = y_proba_oof[:, classes.index('mild')]
proba_severe = y_proba_oof[:, classes.index('severe')]
confidence   = y_proba_oof.max(axis=1)

audit_df['true_label']      = true_label
audit_df['predicted_label'] = predicted_label
audit_df['confidence']      = confidence
audit_df['proba_benign']    = proba_benign
audit_df['proba_mild']      = proba_mild
audit_df['proba_severe']    = proba_severe
audit_df['correct'] = audit_df['true_label'] == audit_df['predicted_label']
audit_df['severity_flip'] = audit_df.apply(
    lambda r: {r['true_label'], r['predicted_label']} == {'benign', 'severe'}, axis=1
)

audit_df = audit_df[[
    'variant_id', 'gene', 'position', 'wt_aa', 'mut_aa',
    'true_label', 'predicted_label', 'confidence',
    'proba_benign', 'proba_mild', 'proba_severe',
    'correct', 'severity_flip',
]]

audit_df.head()

,variant_id,gene,position,wt_aa,mut_aa,true_label,predicted_label,confidence,proba_benign,proba_mild,proba_severe,correct,severity_flip
0,HBB-Gly119Ser,HBB,119,Gly,Ser,mild,mild,0.725,0.085,0.725,0.190,True,False
1,HBB-Leu78Met,HBB,78,Leu,Met,mild,mild,0.865,0.045,0.865,0.090,True,False
2,HBB-Val34Leu,HBB,34,Val,Leu,mild,mild,0.625,0.165,0.625,0.210,True,False
3,HBB-Val54Ile,HBB,54,Val,Ile,mild,mild,0.870,0.005,0.870,0.125,True,False
4,HBB-Asn108Lys,HBB,108,Asn,Lys,severe,mild,0.425,0.210,0.425,0.365,False,False


In [10]:
n_total    = len(audit_df)
n_correct  = int(audit_df['correct'].sum())
n_incorrect = n_total - n_correct
n_flips    = int(audit_df['severity_flip'].sum())

print(f"Total rows: {n_total}")
print(f"Correct:    {n_correct}")
print(f"Incorrect:  {n_incorrect}")
print(f"Severity flips (true/predicted = {{'benign','severe'}}): {n_flips}")

out_path = DATA_DIR / 'full_audit_153_variants.csv'
audit_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path.resolve()}")

Total rows: 153
Correct:    100
Incorrect:  53
Severity flips (true/predicted = {'benign','severe'}): 8

Saved to /Users/aditirajesh/Desktop/hemoscope/notebooks/data/full_audit_153_variants.csv


# Step 4 — Inspect the 8 severity-flip cases

Filter `audit_df` to `severity_flip == True` and display all 8 rows in full,
sorted by `confidence` descending. No feature values or mechanism hypotheses yet.

In [11]:
flip_cols = [
    'variant_id', 'gene', 'position', 'wt_aa', 'mut_aa',
    'true_label', 'predicted_label', 'confidence',
    'proba_benign', 'proba_mild', 'proba_severe',
]

flips_df = (
    audit_df.loc[audit_df['severity_flip'], flip_cols]
    .sort_values('confidence', ascending=False)
    .reset_index(drop=True)
)

print(f"Severity-flip rows: {len(flips_df)}")

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
    print(flips_df.to_string(index=False))

Severity-flip rows: 8
    variant_id gene  position wt_aa mut_aa true_label predicted_label  confidence  proba_benign  proba_mild  proba_severe
 HBA1-Phe43Leu HBA1        43   Phe    Leu     severe          benign       0.710          0.71       0.160         0.130
 HBA1-Gly57Asp HBA1        57   Gly    Asp     benign          severe       0.615          0.02       0.365         0.615
 HBA1-Arg31Lys HBA1        31   Arg    Lys     severe          benign       0.570          0.57       0.390         0.040
HBA1-Lys127Asn HBA1       127   Lys    Asn     benign          severe       0.505          0.04       0.455         0.505
  HBA1-Lys7Asn HBA1         7   Lys    Asn     benign          severe       0.500          0.12       0.380         0.500
HBA1-Ala111Val HBA1       111   Ala    Val     benign          severe       0.495          0.20       0.305         0.495
 HBA1-Gly59Arg HBA1        59   Gly    Arg     severe          benign       0.420          0.42       0.170         0.410
 H

# Step 5 — Gene-stratified error rates

Is HBA1 simply small, or disproportionately harder/noisier? Stratify accuracy,
severity-flip rate, and macro-F1 by gene using the existing `audit_df`
(`true_label`/`predicted_label`/`correct`/`severity_flip` columns — no new
feature pulls, no narrative yet).

In [12]:
print("1. Gene distribution (full 153-row training set):")
gene_counts = audit_df['gene'].value_counts()
print(gene_counts.to_string())

print("\n2. Gene-stratified accuracy (fraction correct == True):")
gene_accuracy = audit_df.groupby('gene')['correct'].mean()
print(gene_accuracy.to_string())

print("\n3. Gene-stratified severity_flip rate (fraction severity_flip == True):")
gene_flip_rate = audit_df.groupby('gene')['severity_flip'].mean()
gene_flip_count = audit_df.groupby('gene')['severity_flip'].sum()
for gene in gene_counts.index:
    print(f"  {gene}: {int(gene_flip_count[gene])}/{int(gene_counts[gene])} = {gene_flip_rate[gene]:.4f}")

print("\n4. Gene-stratified macro-F1 (on true_label/predicted_label, per-gene subset):")
for gene in gene_counts.index:
    subset = audit_df[audit_df['gene'] == gene]
    gene_macro_f1 = f1_score(subset['true_label'], subset['predicted_label'], average='macro')
    print(f"  {gene}: n={len(subset)}, macro-F1={gene_macro_f1:.4f}")

1. Gene distribution (full 153-row training set):
gene
HBA1    99
HBB     54

2. Gene-stratified accuracy (fraction correct == True):
gene
HBA1    0.616162
HBB     0.722222

3. Gene-stratified severity_flip rate (fraction severity_flip == True):
  HBA1: 8/99 = 0.0808
  HBB: 0/54 = 0.0000

4. Gene-stratified macro-F1 (on true_label/predicted_label, per-gene subset):
  HBA1: n=99, macro-F1=0.4947
  HBB: n=54, macro-F1=0.4015


# Step 6 — Raw feature values for the 8 HBA1 flips vs. correctly-classified HBA1 variants

Three things, no narrative:
1. Full 9-feature table for the 8 flip cases.
2. Mean/median of each feature for (a) HBA1 true=benign & correct, (b) HBA1 true=severe & correct, (c) the 8 flips — to see if flips are outliers on any single feature.
3. AF2 structural-model reliability check for HBA1 vs HBB: modelled chain length, distinct training positions used, and `af2_confidence` (pLDDT) distribution across all rows per gene (not just flips).

In [13]:
flip_variant_ids = audit_df.loc[audit_df['severity_flip'], 'variant_id']

flip_features_df = (
    df[df['variant_id'].isin(flip_variant_ids)][['variant_id'] + FEATURE_COLS]
    .merge(audit_df[['variant_id', 'true_label', 'predicted_label', 'confidence']], on='variant_id')
    .sort_values('confidence', ascending=False)
    .reset_index(drop=True)
)

display_cols = ['variant_id', 'true_label', 'predicted_label', 'confidence'] + FEATURE_COLS
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
    print(flip_features_df[display_cols].to_string(index=False))

    variant_id true_label predicted_label  confidence  blosum62_score  conservation_score  pssm_score  position_entropy  heme_distance_angstrom  burial_binary  is_interface_residue  af2_confidence  contact_count_wt
 HBA1-Phe43Leu     severe          benign       0.710               0              0.4594      -0.747            2.3366                  121.79              0                 False            98.7                 6
 HBA1-Gly57Asp     benign          severe       0.615              -1              0.4768      -3.536            2.2613                  122.52              1                 False            98.8                13
 HBA1-Arg31Lys     severe          benign       0.570               2              0.2781      -1.555            3.1198                  116.30              1                 False            98.9                11
HBA1-Lys127Asn     benign          severe       0.505               0              0.4507      -0.963            2.3742                  101

In [14]:
# HBA1-only frame combining audit_df's labels/flags with df's raw feature values
hba1_full = audit_df[audit_df['gene'] == 'HBA1'][['variant_id', 'true_label', 'correct', 'severity_flip']].merge(
    df[['variant_id'] + FEATURE_COLS], on='variant_id'
)

group_benign_correct = hba1_full[(hba1_full['true_label'] == 'benign') & (hba1_full['correct'])]
group_severe_correct = hba1_full[(hba1_full['true_label'] == 'severe') & (hba1_full['correct'])]
group_flips           = hba1_full[hba1_full['severity_flip']]

print(f"(a) HBA1 true=benign, correct=True: n={len(group_benign_correct)}")
print(f"(b) HBA1 true=severe, correct=True: n={len(group_severe_correct)}")
print(f"(c) HBA1 severity-flip cases:       n={len(group_flips)}")

summary = pd.DataFrame({
    'benign_correct_mean':   group_benign_correct[FEATURE_COLS].mean(),
    'benign_correct_median': group_benign_correct[FEATURE_COLS].median(),
    'severe_correct_mean':   group_severe_correct[FEATURE_COLS].mean(),
    'severe_correct_median': group_severe_correct[FEATURE_COLS].median(),
    'flips_mean':            group_flips[FEATURE_COLS].mean(),
    'flips_median':          group_flips[FEATURE_COLS].median(),
})

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
    print(summary.round(4).to_string())

(a) HBA1 true=benign, correct=True: n=7
(b) HBA1 true=severe, correct=True: n=6
(c) HBA1 severity-flip cases:       n=8
                        benign_correct_mean  benign_correct_median  severe_correct_mean  severe_correct_median  flips_mean  flips_median
blosum62_score                       0.0000                 0.0000              -1.1667                -1.5000     -0.3750        0.0000
conservation_score                   0.2809                 0.2784               0.6215                 0.6096      0.5186        0.5101
pssm_score                          -0.1683                 0.2710              -2.3313                -2.5360     -1.1879       -1.5485
position_entropy                     3.1080                 3.1186               1.6359                 1.6876      2.0808        2.1174
heme_distance_angstrom             117.9743               121.8400             116.1050               122.5750    113.6750      119.0450
burial_binary                        0.0000               

In [15]:
import warnings
from Bio.PDB import PDBParser

warnings.filterwarnings('ignore')  # AF2 PDBs lack some optional records; Biopython warns harmlessly
parser = PDBParser(QUIET=True)

af2_paths = {'HBA1': DATA_DIR / 'AF2_P69905.pdb', 'HBB': DATA_DIR / 'AF2_P68871.pdb'}
af2_residue_counts = {}
for gene, path in af2_paths.items():
    structure = parser.get_structure(gene, str(path))
    chain = list(structure[0].get_chains())[0]
    residues = [r for r in chain if r.id[0] == ' ']
    af2_residue_counts[gene] = len(residues)

print("AF2 model chain length (total residues modelled):")
for gene, n in af2_residue_counts.items():
    print(f"  {gene}: {n} residues")

print("\nDistinct training positions used per gene (out of total AF2 chain length):")
for gene in ['HBA1', 'HBB']:
    n_positions = df[df['gene'] == gene]['position'].nunique()
    n_rows = int((df['gene'] == gene).sum())
    print(f"  {gene}: {n_positions} distinct positions ({n_rows} training rows) out of {af2_residue_counts[gene]} modelled residues")

print("\naf2_confidence (pLDDT) across ALL rows per gene (not just flips):")
af2_by_gene = df.groupby('gene')['af2_confidence'].agg(['count', 'mean', 'median', 'min', 'max', 'std'])
print(af2_by_gene.round(4).to_string())

AF2 model chain length (total residues modelled):
  HBA1: 142 residues
  HBB: 147 residues

Distinct training positions used per gene (out of total AF2 chain length):
  HBA1: 70 distinct positions (99 training rows) out of 142 modelled residues
  HBB: 48 distinct positions (54 training rows) out of 147 modelled residues

af2_confidence (pLDDT) across ALL rows per gene (not just flips):
      count     mean  median   min   max     std
gene                                            
HBA1     99  98.3919    98.6  96.4  98.9  0.5233
HBB      54  97.4407    98.2  79.3  98.9  2.8420


# Step 7 — Were the 10 deployed variants part of the training set?

If a variant shown in the deployed app (`backend/data/variants.py`) was also a row
in `training_variants.csv`, then its `severity`/`confidence` in the app was not an
independent held-out prediction — it (or a near-identical row) was part of what the
classifier was fit on. Checking this directly by matching on
`gene + position + wt_aa + mut_aa`.

First confirm the amino-acid code format actually used in the merged training
dataframe before matching — not assumed.

In [16]:
# Confirm the actual amino-acid code format in labels_df (don't assume 3-letter vs 1-letter)
print("wt_aa sample values:", list(labels_df['wt_aa'].unique()[:5]))
print("mut_aa sample values:", list(labels_df['mut_aa'].unique()[:5]))
aa_format_is_three_letter = labels_df['wt_aa'].str.len().eq(3).all()
print(f"\nwt_aa is consistently 3-letter codes: {aa_format_is_three_letter}")

# audit_df (and df) don't carry wt_aa/mut_aa themselves beyond what was merged in
# Step 3 — pull them again here from labels_df, keyed on variant_id, alongside
# the OOF prediction columns already in audit_df.
training_lookup = labels_df[['variant_id', 'gene', 'position', 'wt_aa', 'mut_aa']].merge(
    audit_df[['variant_id', 'true_label', 'predicted_label', 'confidence']],
    on='variant_id', how='inner'  # inner: only rows that survived the dropna in Step 1
)
print(f"\ntraining_lookup rows: {len(training_lookup)} (should be 153)")

wt_aa sample values: ['Gly', 'Leu', 'Val', 'Asn', 'Phe']
mut_aa sample values: ['Ser', 'Met', 'Leu', 'Ile', 'Lys']

wt_aa is consistently 3-letter codes: True

training_lookup rows: 153 (should be 153)


In [17]:
# The 10 variants currently deployed in backend/data/variants.py.
# wt_aa/mut_aa given here in 3-letter form to match the confirmed labels_df format above.
deployed_variants = [
    {"variant_id": "HBB-Glu6Val",    "gene": "HBB",  "position": 6,   "wt_aa": "Glu", "mut_aa": "Val"},
    {"variant_id": "HBB-Glu26Lys",   "gene": "HBB",  "position": 26,  "wt_aa": "Glu", "mut_aa": "Lys"},
    {"variant_id": "HBB-Lys17Asn",   "gene": "HBB",  "position": 17,  "wt_aa": "Lys", "mut_aa": "Asn"},
    {"variant_id": "HBA1-Asp74His",  "gene": "HBA1", "position": 74,  "wt_aa": "Asp", "mut_aa": "His"},
    {"variant_id": "HBB-Val98Met",   "gene": "HBB",  "position": 98,  "wt_aa": "Val", "mut_aa": "Met"},
    {"variant_id": "HBB-Glu121Gln",  "gene": "HBB",  "position": 121, "wt_aa": "Glu", "mut_aa": "Gln"},
    {"variant_id": "HBB-Glu121Lys",  "gene": "HBB",  "position": 121, "wt_aa": "Glu", "mut_aa": "Lys"},
    {"variant_id": "HBA1-Ala120Thr", "gene": "HBA1", "position": 120, "wt_aa": "Ala", "mut_aa": "Thr"},
    {"variant_id": "HBA1-Asp47His",  "gene": "HBA1", "position": 47,  "wt_aa": "Asp", "mut_aa": "His"},
    {"variant_id": "HBB-Asn102Thr",  "gene": "HBB",  "position": 102, "wt_aa": "Asn", "mut_aa": "Thr"},
]

print(f"{'deployed variant_id':<16} {'in training?':<13} {'true_label':<11} {'predicted_label':<16} {'confidence'}")
print("-" * 75)
for v in deployed_variants:
    match = training_lookup[
        (training_lookup['gene'] == v['gene']) &
        (training_lookup['position'] == v['position']) &
        (training_lookup['wt_aa'] == v['wt_aa']) &
        (training_lookup['mut_aa'] == v['mut_aa'])
    ]
    if len(match) == 0:
        print(f"{v['variant_id']:<16} {'NO':<13} {'-':<11} {'-':<16} {'-'}")
    else:
        row = match.iloc[0]
        flag = "" if row['variant_id'] == v['variant_id'] else f"  (matched as {row['variant_id']})"
        print(f"{v['variant_id']:<16} {'YES':<13} {row['true_label']:<11} {row['predicted_label']:<16} {row['confidence']:.4f}{flag}")
        if len(match) > 1:
            print(f"  WARNING: {len(match)} matching rows found for {v['variant_id']}, showing first")

deployed variant_id in training?  true_label  predicted_label  confidence
---------------------------------------------------------------------------
HBB-Glu6Val      NO            -           -                -
HBB-Glu26Lys     YES           severe      mild             0.9050
HBB-Lys17Asn     NO            -           -                -
HBA1-Asp74His    NO            -           -                -
HBB-Val98Met     NO            -           -                -
HBB-Glu121Gln    NO            -           -                -
HBB-Glu121Lys    NO            -           -                -
HBA1-Ala120Thr   NO            -           -                -
HBA1-Asp47His    YES           mild        mild             0.4900
HBB-Asn102Thr    NO            -           -                -


# Step 8 — Raw ClinVar string for HbE, and direct confirmation of the leakage mechanism

Two separate checks:
1. Does `training_variants.csv` preserve a raw ClinVar significance string (not just the mapped `severity_label`)? Print the full row for `HBB-Glu26Lys` verbatim.
2. Confirm directly — not by inference — that `notebook 05` scored HbE using a model object that had HbE's own training label inside its fit data, by reading notebook 04/05's source for which artifact gets fit on what, and by locating HbE's row in this notebook's own `df`/`y`.

In [18]:
# 1. Raw ClinVar metadata check
print("Full column list in raw training_variants.csv:")
print(list(labels_df.columns))

hbe_raw_row = labels_df[labels_df['variant_id'] == 'HBB-Glu26Lys']
print(f"\nFull raw row for HBB-Glu26Lys (every column, exactly as stored):")
with pd.option_context('display.max_colwidth', None):
    print(hbe_raw_row.T.to_string(header=False))

if 'clinvar_significance' in labels_df.columns:
    raw_sig = hbe_raw_row['clinvar_significance'].iloc[0]
    print(f"\nRaw ClinVar significance string for HBB-Glu26Lys, verbatim: \"{raw_sig}\"")
    print(f"This was mapped to severity_label = \"{hbe_raw_row['severity_label'].iloc[0]}\"")
else:
    print("\nNo 'clinvar_significance' (or equivalent) column found — "
          "notebook 01 only kept the mapped severity_label, not the raw ClinVar string. "
          "No review-status or submission-count columns are present either.")

Full column list in raw training_variants.csv:
['variant_id', 'gene', 'position', 'wt_aa', 'mut_aa', 'wt_aa_1', 'mut_aa_1', 'severity_label', 'clinvar_significance', 'uniprot_id']

Full raw row for HBB-Glu26Lys (every column, exactly as stored):
variant_id            HBB-Glu26Lys
gene                           HBB
position                        26
wt_aa                          Glu
mut_aa                         Lys
wt_aa_1                          E
mut_aa_1                         K
severity_label              severe
clinvar_significance    pathogenic
uniprot_id                  P68871

Raw ClinVar significance string for HBB-Glu26Lys, verbatim: "pathogenic"
This was mapped to severity_label = "severe"


In [19]:
# 2. Direct leakage-mechanism confirmation

# (a) HbE's row is in this notebook's own df/y (full 153-row training data, Step 1)
hbe_idx = df.index[df['variant_id'] == 'HBB-Glu26Lys']
assert len(hbe_idx) == 1, f"Expected exactly one match, found {len(hbe_idx)}"
hbe_idx = hbe_idx[0]

print(f"HBB-Glu26Lys row index in df/X/y: {hbe_idx}")
print(f"df.loc[{hbe_idx}, 'severity_label'] = '{df.loc[hbe_idx, 'severity_label']}'")
print(f"y[{hbe_idx}] (label-encoded)        = {y[hbe_idx]}  "
      f"(le.inverse_transform -> '{le.inverse_transform([y[hbe_idx]])[0]}')")

# (b) Read notebook 04 and 05 source directly off disk to confirm which artifact
#     gets fit on what, and which artifact notebook 05 loads to score HbE.
with open('04_train_classifier.ipynb') as f:
    nb04 = json.load(f)
with open('05_generate_variants_py.ipynb') as f:
    nb05 = json.load(f)

def find_lines(nb, needle):
    hits = []
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        for line in ''.join(cell['source']).split('\n'):
            if needle in line:
                hits.append(line.strip())
    return hits

print("\nnotebook 04 source — fitting + saving the calibrated model:")
for needle in ['X_imputed = imputer.fit_transform', 'calibrated_rf.fit', 'joblib.dump(calibrated_rf']:
    for line in find_lines(nb04, needle):
        print(f"  {line}")

print("\nnotebook 05 source — loading that exact artifact and using it to score HbE:")
for needle in ["joblib.load(MODELS_DIR / 'hemoscope_rf_calibrated.pkl')", 'clf.predict_proba']:
    for line in find_lines(nb05, needle):
        print(f"  {line}")

print(
    "\nConfirmed: notebook 04 fits `calibrated_rf` via `calibrated_rf.fit(X_imputed, y)` "
    "on X_imputed/y derived from the FULL 153-row df (no held-out split — X_imputed comes "
    "from imputer.fit_transform(X), not a train/test partition), then saves it as "
    "hemoscope_rf_calibrated.pkl. Notebook 05 loads that exact file as `clf` and calls "
    "clf.predict_proba(...) directly on HBB-Glu26Lys's feature vector to produce the "
    "severity/confidence written into variants_ml.py / variants.py.\n"
    "\n"
    "HBB-Glu26Lys's (features, label='severe') pair was inside the data `calibrated_rf` "
    "was fit on. The notebook 05 prediction for this variant is therefore an in-sample "
    "evaluation of a model on a point it was trained on, not a held-out prediction.\n"
    "\n"
    "YES — this is leakage by construction, not coincidence."
)

HBB-Glu26Lys row index in df/X/y: 23
df.loc[23, 'severity_label'] = 'severe'
y[23] (label-encoded)        = 2  (le.inverse_transform -> 'severe')

notebook 04 source — fitting + saving the calibrated model:
  X_imputed = imputer.fit_transform(X)
  calibrated_rf.fit(X_imputed, y)
  joblib.dump(calibrated_rf, MODELS_DIR / 'hemoscope_rf_calibrated.pkl')

notebook 05 source — loading that exact artifact and using it to score HbE:
  clf     = joblib.load(MODELS_DIR / 'hemoscope_rf_calibrated.pkl')
  proba = clf.predict_proba(X_imp)[0]

Confirmed: notebook 04 fits `calibrated_rf` via `calibrated_rf.fit(X_imputed, y)` on X_imputed/y derived from the FULL 153-row df (no held-out split — X_imputed comes from imputer.fit_transform(X), not a train/test partition), then saves it as hemoscope_rf_calibrated.pkl. Notebook 05 loads that exact file as `clf` and calls clf.predict_proba(...) directly on HBB-Glu26Lys's feature vector to produce the severity/confidence written into variants_ml.py / variant

# Step 9 — Per-variant mechanism notes for the 8 HBA1 severity flips

These are hypotheses consistent with the feature numbers already computed, not
confirmed biological mechanisms — the 9 training features measure sequence
conservation and per-residue structural proximity/burial/interface status; none
of them directly measure protein stability or degradation, which is a known
route to pathogenicity for some alpha-globin variants. Confidence values below
are `max(proba_benign, proba_mild, proba_severe)`; `proba_*` breakdowns are from
Step 4/6.

**HBA1-Phe43Leu** (true=severe, pred=benign, conf=0.710, proba: benign=0.71/mild=0.16/severe=0.13). Nearly every feature here matches the benign-correct group's profile rather than severe-correct's: `blosum62=0` and `pssm=-0.747` sit close to the benign-correct means (0.0, -0.168) and far from severe-correct's (-1.167, -2.331), and `burial=0`/`interface=False`/`contact_count=6` all match the benign-correct profile (means 0.0/0.0/6.57) rather than severe-correct's (0.5/0.167/9.33). The model's confident "benign" call is consistent with essentially the whole feature vector looking benign-like despite the true label being severe — suggestive of a pathogenicity route this feature set doesn't register, possibly stability/degradation-based rather than substitution-chemistry- or contact-based.

**HBA1-Gly57Asp** (true=benign, pred=severe, conf=0.615, proba: benign=0.02/mild=0.365/severe=0.615). The reverse pattern: `pssm=-3.536` is more extreme than the severe-correct mean (-2.331), and `burial=1`/`contact_count=13` exceed the severe-correct means (0.5/9.33) — the highest contact count among all 8 flips. Structurally and by substitution score this variant looks like the most "severe-typical" case in the flip set, yet its true label is benign; this may indicate the buried/high-contact local environment doesn't translate to functional severity for this particular substitution, a distinction the current features can't make.

**HBA1-Arg31Lys** (true=severe, pred=benign, conf=0.570, proba: benign=0.57/mild=0.39/severe=0.04). `blosum62=+2` is the highest (most conservative) substitution score among all 8 flips, exceeding even the benign-correct mean (0.0) — a strong benign-leaning signal — while `burial=1` and `contact_count=11` sit on the severe-correct side (means 0.5, 9.33). The conflicting signal resolved toward benign, but `proba_severe` is the lowest of any flip (0.04), suggesting the model essentially ruled out severity entirely here despite the true label, possibly because a conservative charge-preserving substitution rarely co-occurs with severity in the rest of the training data regardless of structural context.

**HBA1-Lys127Asn** (true=benign, pred=severe, conf=0.505 — the closest to an exact tie among benign/severe, proba: benign=0.04/mild=0.455/severe=0.505). This is the only one of the 8 flips with `is_interface_residue=True`, which sits well above both the benign-correct mean (0.0) and even the severe-correct mean (0.167) for that feature — interface status appears to be doing most of the work pushing this toward "severe." With `proba_mild` almost equal to `proba_severe`, the model is not confidently distinguishing mild from severe here at all; this reads as a genuine decision-boundary case rather than a one-feature-driven error.

**HBA1-Lys7Asn** (true=benign, pred=severe, conf=0.500 — an exact tie, proba: benign=0.12/mild=0.38/severe=0.50). `conservation_score=0.5434` is the highest among all 8 flips and close to the severe-correct mean (0.622), and `burial=1`/`contact_count=12` again sit nearer the severe-correct profile (0.5/9.33) than benign-correct's (0.0/6.57). With the top two probabilities separated by only 0.12 and the top prediction itself at exactly 0.500, this case is effectively within the model's noise floor — the feature values lean mildly severe-typical, but not decisively enough to call it anything other than a coin flip.

**HBA1-Ala111Val** (true=benign, pred=severe, conf=0.495, proba: benign=0.20/mild=0.305/severe=0.495). `position_entropy=1.9076` is low (i.e., evolutionarily constrained), closer to the severe-correct mean (1.636) than the benign-correct mean (3.108), while `burial=0` and `blosum62=0` match the benign-correct profile exactly. The features pull in different directions and land on a three-way near-tie (benign 0.20 / mild 0.31 / severe 0.50) — read together with the conservation/entropy values, this may hint that a structurally tolerant but evolutionarily conserved position contributes disease risk by a mechanism (e.g. subtle stability loss) the burial/contact features don't capture, but the data here are far too close to call that confidently.

**HBA1-Gly59Arg** (true=severe, pred=benign, conf=0.420 — proba: benign=0.42/mild=0.17/severe=0.41, essentially a benign/severe tie). Unlike the other flips, this variant's feature profile largely matches the severe-correct group: `blosum62=-2` and `pssm=-3.536` are at or beyond the severe-correct means (-1.167, -2.331), `position_entropy=1.0797` is the lowest (most conserved) of all 8 flips and below the severe-correct mean (1.636), and `burial=1` matches it too. Despite this severe-typical profile correctly pointing toward the true label, the model's output is a near-exact tie that fell on the wrong side — this looks less like a feature blind spot and more like a borderline case sitting right on the decision boundary, where small amounts of noise in the calibrated probabilities are enough to flip the argmax.

**HBA1-Arg92Leu** (true=severe, pred=benign, conf=0.420 — proba: benign=0.42/mild=0.405/severe=0.175, the *severe* class barely registers at all). `blosum62=-2` matches the severe-correct mean direction, but `pssm=+3.931` is the single most extreme value (in either direction) across all 8 flips and sits far on the opposite side from the severe-correct mean (-2.331) — a strong internal conflict between two sequence-derived features. `af2_confidence=97.5` is also the lowest among the 8 flips, though still high in absolute terms. With `proba_severe` the lowest of any flip case, the model doesn't seem to register this position as severity-associated under any of the 9 features, regardless of the conflicting blosum/pssm signals — most of its uncertainty is actually a benign-vs-mild question, with "severe" essentially ruled out.

---

**Aggregate pattern.** 8 of 153 variants (5.2%) and 8 of the 53 total misclassifications (15.1%) are severity flips — predictions landing on the opposite extreme from the true label. All 8 are concentrated in HBA1 (8/99 = 8.1% of HBA1 rows); HBB has zero (0/54 = 0.0%). This concentration is not explained by HBA1 being a smaller or lower-quality subset: HBA1 has more training rows than HBB (99 vs 54), a higher per-gene macro-F1 (0.4947 vs 0.4015), and AF2 pLDDT confidence that is on average *higher* and *less variable* than HBB's (mean 98.39/std 0.52 vs mean 97.44/std 2.84). Across all 8 flip cases, feature values sit numerically between the benign-correct and severe-correct group means/medians on most features rather than at an extreme on either side — these read as genuine boundary cases the model's decision surface mishandles, not as outliers or data-quality artifacts. One plausible, unconfirmed hypothesis consistent with these numbers: the 9 training features (sequence conservation + per-residue structural proximity/burial/interface) may have less separating power specifically for whichever subset of alpha-globin pathogenicity routes act through protein stability or degradation rather than heme-pocket disruption or subunit-interface contact — but no feature in this dataset directly measures stability or degradation, so this cannot be confirmed from the data on hand and should be treated as a hypothesis to test, not a finding.

# Step 10 — clinvar_significance vs severity_label cross-tabulation

Documented mapping (`notebooks/01_fetch_training_data.ipynb`, `SIGNIFICANCE_MAP`):
```
'benign': 'benign',
'likely benign': 'benign',
'uncertain significance': 'mild',
'likely pathogenic': 'severe',
'pathogenic': 'severe',
'pathogenic/likely pathogenic': 'severe',
'benign/likely benign': 'benign',
```
Note `'uncertain significance'` (VUS) maps to `'mild'`, not to a dedicated "unknown" bucket — checking how much of the `mild` class this accounts for.

In [20]:
# Cross-tab clinvar_significance vs severity_label, full 153-row raw labels_df
crosstab = pd.crosstab(labels_df['clinvar_significance'], labels_df['severity_label'])
crosstab['total'] = crosstab.sum(axis=1)
crosstab = crosstab.sort_values('total', ascending=False)

print("clinvar_significance vs severity_label (153 rows):")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
    print(crosstab.to_string())

# Confirm many-to-one: each raw clinvar_significance maps to exactly one severity_label
n_labels_per_sig = labels_df.groupby('clinvar_significance')['severity_label'].nunique()
clean_mapping = (n_labels_per_sig == 1).all()
print(f"\nEach raw clinvar_significance maps to exactly one severity_label (clean many-to-one): {clean_mapping}")
if not clean_mapping:
    print("Significance strings mapping to >1 severity_label:")
    print(n_labels_per_sig[n_labels_per_sig > 1])

clinvar_significance vs severity_label (153 rows):
severity_label                benign  mild  severe  total
clinvar_significance                                     
uncertain significance             0   106       0    106
likely benign                     18     0       0     18
likely pathogenic                  0     0      18     18
pathogenic                         0     0       6      6
pathogenic/likely pathogenic       0     0       3      3
benign                             1     0       0      1
benign/likely benign               1     0       0      1



Each raw clinvar_significance maps to exactly one severity_label (clean many-to-one): True


In [21]:
# VUS share of the 'mild' class specifically
mild_rows = labels_df[labels_df['severity_label'] == 'mild']
n_mild = len(mild_rows)
n_mild_vus = (mild_rows['clinvar_significance'] == 'uncertain significance').sum()
n_mild_other = n_mild - n_mild_vus

print(f"Of the {n_mild} variants with severity_label == 'mild':")
print(f"  clinvar_significance == 'uncertain significance' (VUS): {n_mild_vus}  ({n_mild_vus / n_mild:.1%})")
print(f"  other raw clinvar_significance values:                  {n_mild_other}  ({n_mild_other / n_mild:.1%})")

print("\nBreakdown of 'other' raw values within the mild class:")
print(mild_rows.loc[mild_rows['clinvar_significance'] != 'uncertain significance', 'clinvar_significance']
      .value_counts().to_string())

Of the 106 variants with severity_label == 'mild':
  clinvar_significance == 'uncertain significance' (VUS): 106  (100.0%)
  other raw clinvar_significance values:                  0  (0.0%)

Breakdown of 'other' raw values within the mild class:
Series([], )


In [22]:
# Raw clinvar_significance for the 8 flip variants + the 1 leakage-affected variant (HbE)
flip_ids = list(flip_variant_ids)  # from Step 6
ids_of_interest = flip_ids + ['HBB-Glu26Lys']

lookup = (
    labels_df[labels_df['variant_id'].isin(ids_of_interest)][['variant_id', 'clinvar_significance']]
    .merge(audit_df[['variant_id', 'true_label', 'predicted_label', 'severity_flip']], on='variant_id')
)
# Order: 8 flips first (by variant_id to match earlier tables), then HbE
lookup['is_flip'] = lookup['variant_id'].isin(flip_ids)
lookup = pd.concat([
    lookup[lookup['variant_id'] != 'HBB-Glu26Lys'],
    lookup[lookup['variant_id'] == 'HBB-Glu26Lys'],
]).reset_index(drop=True)

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
    print(lookup.to_string(index=False))

print(f"\nOf the 8 severity-flip variants, {(lookup.loc[lookup['is_flip'], 'clinvar_significance'] == 'uncertain significance').sum()} "
      f"have clinvar_significance == 'uncertain significance' (VUS).")
print(f"HBB-Glu26Lys (HbE, leakage case) clinvar_significance: "
      f"'{lookup.loc[lookup['variant_id'] == 'HBB-Glu26Lys', 'clinvar_significance'].iloc[0]}'")

    variant_id         clinvar_significance true_label predicted_label  severity_flip  is_flip
 HBA1-Arg31Lys            likely pathogenic     severe          benign           True     True
 HBA1-Gly59Arg pathogenic/likely pathogenic     severe          benign           True     True
HBA1-Ala111Val                likely benign     benign          severe           True     True
  HBA1-Lys7Asn                likely benign     benign          severe           True     True
 HBA1-Gly57Asp                likely benign     benign          severe           True     True
HBA1-Lys127Asn                likely benign     benign          severe           True     True
 HBA1-Phe43Leu            likely pathogenic     severe          benign           True     True
 HBA1-Arg92Leu            likely pathogenic     severe          benign           True     True
  HBB-Glu26Lys                   pathogenic     severe            mild          False    False

Of the 8 severity-flip variants, 0 have clinvar_s